In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.io
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

ROOT_DIR     = 'pastovus sukiai'
OUTPUT_PATH  = 'signals2.parquet'
MAT_FILENAME = '5.mat'

FEATS   = [f'Feat{i}' for i in range(7)]
BATCHES = ['1 bandymas', '2 bandymas']
LOADS   = ['25', '50', '75', '100', '125']
RPMS    = ['1000', '2000', '3000', '4000', '5000']

print(f'Root direktorija : {os.path.abspath(ROOT_DIR)}')

In [ ]:
records = []
missing = []
errors  = []

for feat in FEATS:
    for batch in BATCHES:
        for load in LOADS:
            for rpm in RPMS:
                mat_path = os.path.join(ROOT_DIR, feat, batch, load, rpm, MAT_FILENAME)

                if not os.path.isfile(mat_path):
                    missing.append(mat_path)
                    continue

                try:
                    mat = scipy.io.loadmat(mat_path)
                    data_key = [k for k in mat.keys() if not k.startswith('__')][0]
                    signal = mat[data_key].squeeze().astype(np.float64)
                    assert signal.ndim == 1, f'Expected 1-D array, got shape {signal.shape}'

                    records.append({
                        'Feature'     : feat,
                        'Bandymas'    : int(batch[0]),
                        'Apkrova(Nm)' : int(load),
                        'Sukiai(rpm)' : int(rpm),
                        'signal'      : signal,
                    })

                except Exception as e:
                    errors.append((mat_path, str(e)))

print(f'Uzkrauta  : {len(records)} signalu')
print(f'Truksta : {len(missing)} failo/u')
print(f'Klaidu  : {len(errors)}')

if missing:
    print('\nTruksta:')
    for p in missing: print(f'  {p}')
if errors:
    print('\nKlaidos:')
    for p, e in errors: print(f'  {p}  ->  {e}')

In [ ]:
dfs = []
for r in records:
    dfs.append(pd.DataFrame({
        'Feature'     : r['Feature'],
        'Bandymas'    : r['Bandymas'],
        'Apkrova(Nm)' : r['Apkrova(Nm)'],
        'Sukiai(rpm)' : r['Sukiai(rpm)'],
        'Value'       : r['signal'],
    }))

df = pd.concat(dfs, ignore_index=True)
df['Feature']     = df['Feature'].astype('category')
df['Bandymas']    = df['Bandymas'].astype(np.int8)
df['Apkrova(Nm)'] = df['Apkrova(Nm)'].astype(np.int16)
df['Sukiai(rpm)'] = df['Sukiai(rpm)'].astype(np.int16)

df.to_parquet(OUTPUT_PATH, index=False)

size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f'Issaugota i : {OUTPUT_PATH}  ({size_mb:.1f} MB)')
print(f'Forma : {df.shape[0]:,} rows x {df.shape[1]} columns')

## Vizualizacija

In [ ]:
N_SHOW = 2048

fig, axes = plt.subplots(7, 1, figsize=(14, 14), sharex=True)
for i, feat in enumerate(FEATS):
    mask = (
        (df['Feature']     == feat) &
        (df['Bandymas']    == 1) &
        (df['Apkrova(Nm)'] == 50) &
        (df['Sukiai(rpm)'] == 1000)
    )
    sig = df[mask]['Value'].values[:N_SHOW]
    axes[i].plot(sig, linewidth=0.6, color=f'C{i}')
    axes[i].set_ylabel(feat, fontsize=9)
    axes[i].grid(True, alpha=0.3)

axes[-1].set_xlabel('Sample index')
fig.suptitle('Raw signals by fault class — Batch 1, 50 Nm, 1000 RPM', fontsize=12)
plt.tight_layout()
plt.savefig('signals_by_class.png', dpi=150, bbox_inches='tight')
plt.show()